# Chapter 1: NumPy Fundamentals

### Why This Matters

NumPy stands for "Numerical Python," and its core contribution is deceptively simple-sounding: doing math on entire arrays of numbers at once, instead of looping over them one at a time. That single idea — **vectorization** — is why a NumPy operation on a million numbers takes milliseconds instead of the seconds or minutes a pure-Python loop would need, and it's the reason literally every numeric Python library exists in the form it does today.

### 1.1 Creating Arrays

In [ ]:
import numpy as np

# From a list
sales = np.array([125, 98, 112, 145, 130, 160])
print(sales)
print(type(sales))     # <class 'numpy.ndarray'>
print(sales.dtype)     # int64 (or int32 on some systems) — the SINGLE data type shared by every element

That last line is the critical difference from a Python list: a NumPy array's `dtype` is uniform across every element. A Python list can freely mix an int, a string, and a float in the same list (as you saw in Volume 01); a NumPy array cannot — every element is stored as the same fixed-size type, packed contiguously in memory. This constraint is *exactly* what makes NumPy fast: the CPU can process a tightly-packed block of same-sized numbers far more efficiently than it can chase down a list of scattered Python objects.

In [ ]:
# Multi-dimensional arrays (matrices)
region_month = np.array([
    [125, 130, 140],   # North
    [98, 105, 110],    # South
    [112, 115, 125],   # East
    [145, 150, 160]    # West
])
print(region_month)
print("Shape:", region_month.shape)    # (4, 3) -> (rows, columns)

`.shape` is one of the first things to check on any array you receive — it tells you the dimensionality (here, 2D: 4 rows × 3 columns) and is the single most common source of "why doesn't this operation work" errors when two arrays don't line up as you expected.

In [ ]:
# Special/generator arrays you'll use constantly
print(np.zeros((3, 4)))                          # a 3x4 array of all zeros — common as a starting "canvas"
print(np.ones((2, 5)))                           # a 2x5 array of all ones
print(np.arange(0, 11, 2))                       # [0 2 4 6 8 10] -- like Python's range(), but returns an array
print(np.linspace(0, 1, 5))                      # 5 evenly spaced numbers from 0 to 1 INCLUSIVE
print(np.random.rand(3, 3))                      # random floats in [0, 1), shape (3,3)
print(np.random.randint(50, 150, size=(3, 4)))   # random integers in [50, 150), shape (3,4)

💡 `np.arange` vs `np.linspace`: `arange` lets you specify the **step size** between values (and, like `range()`, excludes the endpoint); `linspace` lets you specify **how many points you want** and includes both endpoints by default. Use `arange` when you know the increment (e.g., "every 2 units"); use `linspace` when you know the count (e.g., "give me exactly 50 evenly spaced points for a smooth chart").

### 1.2 Vectorization — NumPy's Superpower

In [ ]:
sales = np.array([125, 98, 112, 145, 130, 160])
targets = np.array([120, 100, 110, 140, 125, 150])

# Element-wise operations — no loop required, ever
print("Difference   :", sales - targets)
print("Achievement %:", (sales / targets) * 100)
print("Growth +10%  :", sales * 1.1)

Compare this to what the equivalent pure-Python loop would look like:

In [ ]:
# The "before NumPy" way — don't do this, but understand what NumPy is replacing
differences = []
for s, t in zip(sales, targets):
    differences.append(s - t)

The NumPy version isn't just shorter — it's a fundamentally different execution model. `sales - targets` applies the subtraction to every corresponding pair of elements internally, in compiled C code, without Python's per-iteration interpreter overhead. This is why the source notebooks call vectorization "the superpower": as your data grows from 6 elements to 6 million, the vectorized version's runtime grows roughly linearly and stays fast, while a naive Python loop's overhead compounds.

In [ ]:
# Aggregations — collapsing an array down to a single summary number
print("Total sales :", sales.sum())
print("Average     :", sales.mean())
print("Max         :", sales.max())
print("Min         :", sales.min())
print("Std Dev     :", sales.std())
print("Variance    :", sales.var())

### 1.3 Indexing, Slicing, and Boolean Masking

In [ ]:
print(sales[0])        # single element
print(sales[1:4])      # slice — same [start:stop) rule from Volume 01's strings/lists
print(sales[-3:])      # last three elements

# Boolean masking — arguably the single most important NumPy skill for an analyst
print(sales > 120)                              # [ True False False  True  True  True] -- a BOOLEAN ARRAY
print(sales[sales > 120])                       # using that boolean array to FILTER -> only the high performers
print(sales[(sales > 100) & (sales < 140)])     # combine conditions with & (and) / | (or) — NOT `and`/`or`

⚠️ **This is the single most important gotcha in this entire chapter:** inside a NumPy boolean condition, you must use `&` and `|`, never Python's `and`/`or` keywords, and you must wrap each condition in its own parentheses: `(sales > 100) & (sales < 140)`. Python's `and`/`or` are designed to work on single `True`/`False` values, not on entire arrays of them, and will raise a confusing `ValueError` ("truth value of an array is ambiguous") if you try. This exact same rule reappears, unchanged, in Pandas filtering later in this volume — learn it once, here.

**Boolean masking is the mechanism behind essentially every "filter my data" operation** you'll do for the rest of this course — `sales[sales > 120]` reads as "give me only the elements of `sales` where the corresponding position in the boolean array is `True`." Internalizing this pattern now means Pandas' `df[df["Sales"] > 120000]` later in this volume will feel completely familiar rather than new syntax to memorize.

In [ ]:
# 2D indexing
print(region_month[0])           # first row (North) -> array([125, 130, 140])
print(region_month[:, 1])        # ':' means "all rows"; column index 1 -> all regions' 2nd month
print(region_month[1:3, 0:2])    # a sub-matrix: rows 1-2, columns 0-1

The comma inside `[]` separates dimensions: `array[rows, columns]`. `:` alone means "every index along this dimension." This generalizes to as many dimensions as your array has (3D arrays — common in image data and deep learning — use the same idea with a third comma-separated slot).

### 1.4 Reshaping and Axis-Wise Functions

In [ ]:
arr = np.arange(12)
print(arr.reshape(3, 4))     # reinterpret the same 12 numbers as a 3x4 grid
print(arr.reshape(2, 6))     # or as a 2x6 grid — same data, different shape

# Statistical functions along a specific AXIS
print("Row-wise mean (each region):", region_month.mean(axis=1))    # axis=1 -> collapse ACROSS columns, one result per row
print("Column-wise mean (each month):", region_month.mean(axis=0))  # axis=0 -> collapse ACROSS rows, one result per column

**The `axis` parameter is the concept beginners get backwards most often**, so here's a mental model that sticks: `axis=0` means "walk down the rows" (so the result collapses the row dimension, giving you one number per **column**); `axis=1` means "walk across the columns" (collapsing the column dimension, giving you one number per **row**). If you ever get this backwards, just check the *shape* of your result — `region_month.mean(axis=1)` on a (4,3) array gives you 4 numbers (one per region/row); `axis=0` gives you 3 numbers (one per month/column). This exact same `axis` argument reappears constantly in Pandas (`df.mean(axis=0)`) and in every deep learning framework in Volume 08.

### 1.5 Advanced NumPy: Broadcasting, Linear Algebra, and Simulation

**Broadcasting** is NumPy's rule set for how arrays of *different* shapes are allowed to interact in an element-wise operation, without you needing to manually resize one of them first:

In [ ]:
a = np.array([10, 20, 30, 40], dtype=float)
b = np.array([1, 2, 3, 4], dtype=float)
print(a / b)          # element-wise: [10. 10. 10. 10.]
print(a * 1.15)        # a single scalar (1.15) is "broadcast" across every element of a

`a * 1.15` is broadcasting in its simplest form: the scalar `1.15` conceptually stretches to match `a`'s shape before the multiplication happens, without NumPy actually copying it in memory. The same principle scales up to genuinely useful business scenarios: converting an entire column of prices from INR to USD with `prices * exchange_rate` in one line, or subtracting a single row of column averages from every row of a table.

**Linear algebra** — NumPy's `linalg` submodule handles matrix operations you'll need again the moment you touch regression (Volume 04) or neural networks (Volume 08):

In [ ]:
A = np.array([[3, 1], [1, 2]], float)
y = np.array([9, 8], float)

x = np.linalg.solve(A, y)     # solves the system of linear equations A·x = y
print("solution:", x)

print("det:", np.linalg.det(A))    # the determinant of A

`np.linalg.solve` is precisely what's happening under the hood when scikit-learn fits a linear regression — the "normal equation" used to find the best-fit coefficients is a linear system, solved with exactly this kind of machinery.

**Random simulation** — reproducible randomness for Monte Carlo-style business modeling:

In [ ]:
rng = np.random.default_rng(42)     # the MODERN way to generate random numbers (prefer this over np.random.seed)

# Estimate the probability that the mean of 50 standard-normal draws exceeds 0.3
samples = rng.normal(size=(10000, 50))
means = samples.mean(axis=1)
probability = (means > 0.3).mean()     # note: booleans sum/average as 1s and 0s, giving you a proportion directly
print(probability)

💡 That last line — `(means > 0.3).mean()` — is a genuinely important idiom: because `True` behaves as `1` and `False` as `0` in arithmetic, taking the `.mean()` of a boolean array directly gives you "the proportion of elements satisfying the condition." This shows up constantly in data analysis (e.g., "what fraction of customers churned?" is `(df["churned"] == True).mean()`).

💡 **Practical note carried over from the source material:** prefer `np.random.default_rng(seed)` over the older, global `np.random.seed(seed)` for any new code. The newer generator API is more explicit (you pass the `rng` object around rather than mutating hidden global state) and avoids subtle bugs where one part of a large program accidentally resets another part's randomness.

### Business Mini-Case: Portfolio Return Simulation

In [ ]:
rng = np.random.default_rng(42)
monthly_returns = rng.normal(loc=0.01, scale=0.04, size=60)   # 5 years of simulated monthly returns

cumulative = np.cumprod(1 + monthly_returns)      # running product -> compounds returns month over month
final_value = 1_000_000 * cumulative[-1]

print(f"Starting Capital : ₹10,00,000")
print(f"Final Value      : ₹{final_value:,.0f}")
print(f"Total Return     : {(final_value/1_000_000 - 1)*100:.1f}%")
print(f"Best Month       : {monthly_returns.max()*100:.1f}%")
print(f"Worst Month      : {monthly_returns.min()*100:.1f}%")

`np.cumprod` (cumulative product) is exactly the right tool whenever a quantity compounds over time — investment returns, population growth, compound interest. `cumulative[-1]` (the last element) gives you the final compounded multiplier after all 60 months.

### Cheat Sheet — NumPy

| Task | Code |
|---|---|
| Create from list | `np.array([1,2,3])` |
| Zeros/ones/range | `np.zeros((r,c))`, `np.ones((r,c))`, `np.arange(start,stop,step)` |
| Evenly spaced | `np.linspace(start, stop, n)` |
| Random | `rng = np.random.default_rng(seed); rng.normal(...)`, `rng.integers(lo,hi,size)` |
| Shape / reshape | `arr.shape`, `arr.reshape(r, c)` |
| Aggregate | `.sum() .mean() .max() .min() .std() .var()` |
| Axis-wise aggregate | `.mean(axis=0)` (per column), `.mean(axis=1)` (per row) |
| Filter | `arr[arr > x]`, combine with `&` `\|`, never `and`/`or` |
| Linear algebra | `np.linalg.solve(A, y)`, `np.linalg.det(A)` |
| Cumulative | `np.cumsum(arr)`, `np.cumprod(arr)` |

### 🎯 Business Challenge — NumPy

*Task:* 4 regions × 6 months of sales (lakhs). Totals per region/month; region with highest average; MoM growth for West; values above 55; overall mean/median/std.

In [ ]:
import numpy as np

data = np.array([
    [45, 48, 52, 49, 55, 60],   # North
    [38, 40, 42, 41, 45, 48],   # South
    [42, 44, 47, 46, 50, 53],   # East
    [50, 53, 58, 55, 62, 65]    # West
])
regions = ["North", "South", "East", "West"]

total_per_region = data.sum(axis=1)
total_per_month = data.sum(axis=0)
avg_per_region = data.mean(axis=1)
best_region = regions[avg_per_region.argmax()]     # argmax returns the INDEX of the max, not the value itself

west = data[3]
mom_growth = (west[1:] - west[:-1]) / west[:-1] * 100     # each month vs. the PREVIOUS month

high_performance = data[data > 55]

print("Total per region :", dict(zip(regions, total_per_region)))
print("Total per month  :", total_per_month)
print("Best region      :", best_region, f"(avg {avg_per_region.max():.1f} lakhs)")
print("West MoM growth% :", np.round(mom_growth, 1))
print("Values above 55  :", high_performance)
print(f"Overall mean={data.mean():.2f}, median={np.median(data):.2f}, std={data.std():.2f}")

💡 `west[1:] - west[:-1]` is the standard NumPy idiom for "difference between consecutive elements": slicing the array offset by one position against itself. You'll reuse this exact pattern for month-over-month or day-over-day calculations throughout your analytics career (Pandas gives you a dedicated `.diff()` method that does the same thing, covered later in this volume).

---